# Module 2: Tracing

> Self-directed module, ~20 min.

Two things in one notebook:

1. **Generate traces** — run the client research agent against a handful of prompts so we have something to query.
2. **Query traces** — pull runs back with `client.list_runs(...)` and the filter DSL; navigate to the project in the LangSmith UI.

With `LANGSMITH_TRACING=true`, every LLM call, tool call, and state transition is captured automatically. No code changes are required on the agent.


## Setup


In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import os
import time
from datetime import datetime, timedelta, timezone
from langsmith import Client, uuid7

from utils.config import load_active_agent, PROJECT_NAME

# Single source of truth for project name; ensure tracing is on.
os.environ["LANGSMITH_PROJECT"] = PROJECT_NAME
os.environ.setdefault("LANGSMITH_TRACING", "true")

client = Client()
agent = load_active_agent()

print(f"Project:  {PROJECT_NAME}")
print(f"Tracing:  {os.environ.get('LANGSMITH_TRACING')}")
print(f"Endpoint: {os.environ.get('LANGSMITH_ENDPOINT', 'https://api.smith.langchain.com')}")


## Warm-up: Generate traces

Five varied prompts — single-tool, profile-only, and multi-step (profile + research) — so the filter DSL has different trace shapes to chew on in Part 1. Each prompt limits the agent's search budget to keep runs fast.


In [ ]:
warmup_prompts = [
    "What's the latest on Apple (AAPL)? One paragraph, search at most once.",
    'Pull up Smith Family Office. What are their top holdings?',
    'What does TechCorp Treasury hold?',
    'Look up Acme Pension Fund, then research one of their top holdings briefly. Search at most once.',
    'Research recent news on NVDA. Search at most once.',
]

total = 0.0
for q in warmup_prompts:
    cfg = {"configurable": {"thread_id": str(uuid7())}}
    t0 = time.perf_counter()
    result = agent.invoke({"messages": [{"role": "user", "content": q}]}, config=cfg)
    elapsed = time.perf_counter() - t0
    total += elapsed
    print(f"[{elapsed:4.1f}s] {result['messages'][-1].content[:120]}")

print(f"\nTotal: {total:.1f}s ({total/len(warmup_prompts):.1f}s avg)")


## Part 1. Querying Traces

`client.read_project(...)` gives the project URL — open it in a browser to confirm the traces above landed. Subsequent cells pull runs back via `client.list_runs(...)` so they can be inspected programmatically.


In [ ]:
try:
    project = client.read_project(project_name=PROJECT_NAME)
    print(f"Project: {project.name}")
    print(f"View traces: {project.url}")
except Exception as e:
    print(f"Could not read project (this is fine if first run): {e}")


### 1.1 Pull recent traces

Useful filters on `client.list_runs(...)`:

- `project_name=` — scope to one project (required for cross-project queries to be reasonable).
- `start_time=` — only runs after this UTC timestamp.
- `is_root=True` — top-level traces only (skips child LLM / tool / subagent spans).
- `limit=` — cap the result set; default is 100.


In [ ]:
since = datetime.now(timezone.utc) - timedelta(hours=1)

recent_runs = list(client.list_runs(
    project_name=PROJECT_NAME,
    start_time=since,
    is_root=True,
    limit=20,
))

print(f"Found {len(recent_runs)} root run(s) in the last hour\n")
for r in recent_runs[:5]:
    latency = (r.end_time - r.start_time).total_seconds() if r.end_time else None
    print(f"- {r.id}  {r.name:25s}  latency={latency}s  error={r.error is not None}")


### 1.2 Filter DSL

The `filter` argument is a small expression language. Common operators: `eq(field, value)`, `gt(field, value)`, `lt(field, value)`, `and(...)`, `or(...)`. Common fields: `latency`, `status`, `name`, `is_root`, `feedback_key`, `feedback_score`.

Three filters worth knowing for an agent like this one — slow runs (the multi-step ones), errored runs, and runs by name (e.g., just the `research-agent` subagent spans):


In [ ]:
# Slow root runs in the last hour (>10s latency)
slow_runs = list(client.list_runs(
    project_name=PROJECT_NAME,
    start_time=since,
    is_root=True,
    filter="gt(latency, 10)",
    limit=20,
))
print(f"{len(slow_runs)} slow root run(s) (>10s) in the last hour")
for r in slow_runs[:5]:
    latency = (r.end_time - r.start_time).total_seconds()
    print(f"  {r.name:25s}  {latency:.1f}s  {r.id}")


In [ ]:
# Errored runs in the last hour
errored_runs = list(client.list_runs(
    project_name=PROJECT_NAME,
    start_time=since,
    filter='eq(status, "error")',
    limit=20,
))
print(f"{len(errored_runs)} errored run(s) in the last hour")
for r in errored_runs[:5]:
    print(f"  {r.name:25s}  {r.error[:80] if r.error else ''}")


In [ ]:
# Just the research-agent subagent spans — useful for inspecting what the subagent received
subagent_runs = list(client.list_runs(
    project_name=PROJECT_NAME,
    start_time=since,
    filter='eq(name, "research-agent")',
    limit=20,
))
print(f"{len(subagent_runs)} research-agent subagent run(s) in the last hour")
for r in subagent_runs[:5]:
    print(f"  {r.id}  parent={r.parent_run_id}")


### 1.3 Navigate in the UI

Click `project.url` printed above to open the project in LangSmith. The traces tab lists every run; clicking one opens the trace tree (model calls, tool calls, subagent spans). The same filter DSL is available in the UI's filter bar.

<!-- Screenshot placeholder: LangSmith project traces tab with the filter bar visible. -->


## Recap

| Topic | API |
|---|---|
| Generate traces | `LANGSMITH_TRACING=true` + `agent.invoke(...)` |
| List runs | `client.list_runs(project_name=..., start_time=..., is_root=True)` |
| Filter — slow | `filter="gt(latency, 10)"` |
| Filter — errored | `filter='eq(status, "error")'` |
| Filter — by run name | `filter='eq(name, "research-agent")'` |

Next: `03_online_evals.ipynb` — score every new trace as it lands.
